# bottleneck-latent-projection composite — cx4: Latent z -> Linear projection -> reshape -> ConvT/BN/ReLU upsample block

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `bottleneck-latent-projection`, `convtranspose-bn-activation-block`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "bottleneck-latent-projection"
DD_ATOM_IDS = ["bottleneck-latent-projection", "convtranspose-bn-activation-block"]
DD_SUBTOPICS = ["Generative: Bottleneck latent projection", "GAN: ConvT+BN+Activation block"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The DCGAN / VAE-decoder entry sequence is: a flat latent code `z` of shape `(B, latent_dim)` becomes a small 3D feature map `(B, C, H, W)` via a learnable Linear projection + reshape, then the ConvT upsampling stack takes over.

1. **bottleneck-latent-projection** — `nn.Linear(latent_dim, C*H*W)` followed by a reshape to `(B, C, H, W)`. The *only* learnable layer at the bottleneck is the Linear; no activation, no norm at the bottleneck output itself.
2. **convtranspose-bn-activation-block** — the canonical `ConvTranspose2d(k=4,s=2,p=1,bias=False) -> BatchNorm2d -> ReLU` triple that doubles spatial size each application.

**Anatomy.**
```python
class Decoder(nn.Module):
    def __init__(self, latent_dim, base_c, base_hw):
        super().__init__()
        self.base_c, self.base_hw = base_c, base_hw
        # bottleneck-latent-projection: Linear then implicit reshape in forward.
        self.proj = nn.Linear(latent_dim, base_c * base_hw * base_hw)
        # convtranspose-bn-activation-block: 2x upsample, channel halving.
        self.block = nn.Sequential(
            nn.ConvTranspose2d(base_c, base_c // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_c // 2),
            nn.ReLU(inplace=True),
        )
    def forward(self, z):
        h = self.proj(z).view(z.size(0), self.base_c, self.base_hw, self.base_hw)
        return self.block(h)
```

**Why no activation on the projection.** Conceptually the bottleneck IS the latent interface — adding an activation would mean the model can never use the full real line for any latent dimension. The conv block immediately after has its own non-linearity (`ReLU`), so the activation 'budget' isn't wasted.

### Composite Exercise — Latent z -> Linear projection -> reshape -> ConvT/BN/ReLU upsample block

**Atoms exercised together**: `bottleneck-latent-projection`, `convtranspose-bn-activation-block`

Implement `cx4_make_decoder_cls()` — return a `Decoder` class.

Contract:
- `Decoder(latent_dim: int, base_channels: int, base_hw: int)`.
- `super().__init__()` first.
- Store `self.base_c = base_channels` and `self.base_hw = base_hw` (the test reads these to know the post-projection shape).
- `self.proj = nn.Linear(latent_dim, base_channels * base_hw * base_hw)` (atom: bottleneck-latent-projection). NO activation here.
- `self.block = nn.Sequential(ConvTranspose2d(base_channels, base_channels // 2, kernel_size=4, stride=2, padding=1, bias=False), BatchNorm2d(base_channels // 2), ReLU(inplace=True))` (atom: convtranspose-bn-activation-block).
- `forward(self, z)`:
  1. `h = self.proj(z)` — shape `(B, base_c * base_hw**2)`.
  2. Reshape `h` to `(B, base_c, base_hw, base_hw)`.
  3. Return `self.block(h)` — shape `(B, base_c // 2, 2*base_hw, 2*base_hw)`.

The test checks:
- Returned value is a class; instance is an nn.Module.
- Children `proj`, `block` (any order).
- `proj` has `in_features=latent_dim`, `out_features=base_channels*base_hw**2`.
- `block` is `nn.Sequential` with `ConvT2d/BN/ReLU` in order; ConvT `bias is None`.
- Forward shape: `(B, latent_dim)` -> `(B, base_channels//2, 2*base_hw, 2*base_hw)`.
- The output is the same as `block(proj(z).view(...))` manually.
- After the bottleneck Linear, NO activation is applied before the conv block.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx4_make_decoder_cls():
    """Return the Decoder class."""
    raise NotImplementedError

def _test_cx4():
    Decoder = cx4_make_decoder_cls()
    assert isinstance(Decoder, type)

    latent_dim, base_c, base_hw = 100, 64, 4
    dec = Decoder(latent_dim=latent_dim, base_channels=base_c, base_hw=base_hw)
    assert isinstance(dec, nn.Module)

    # Case A: named children.
    kids = dict(dec.named_children())
    assert set(kids.keys()) == {'proj', 'block'}, (
        f"expected children {{'proj','block'}}; got {sorted(kids.keys())}"
    )
    assert isinstance(kids['proj'], nn.Linear)
    assert kids['proj'].in_features == latent_dim
    assert kids['proj'].out_features == base_c * base_hw * base_hw
    assert isinstance(kids['block'], nn.Sequential)
    block_kids = list(kids['block'].children())
    assert len(block_kids) == 3
    assert isinstance(block_kids[0], nn.ConvTranspose2d)
    assert isinstance(block_kids[1], nn.BatchNorm2d)
    assert isinstance(block_kids[2], nn.ReLU)
    assert block_kids[0].bias is None, 'ConvT bias must be None'
    assert block_kids[0].in_channels == base_c and block_kids[0].out_channels == base_c // 2

    # Case B: forward shape.
    dec.eval()
    B = 5
    z = t.randn(B, latent_dim)
    with t.no_grad():
        out = dec(z)
    assert out.shape == (B, base_c // 2, 2 * base_hw, 2 * base_hw), (
        f'expected ({B},{base_c // 2},{2*base_hw},{2*base_hw}); got {tuple(out.shape)}'
    )

    # Case C: matches manual proj+reshape+block path.
    with t.no_grad():
        h_manual = kids['proj'](z).view(B, base_c, base_hw, base_hw)
        expected = kids['block'](h_manual)
    assert t.allclose(out, expected, atol=1e-6), 'forward disagrees with manual proj.view.block'

    # Case D: NO activation between proj and block — proves the bottleneck is bare Linear.
    # Reasoning: if a ReLU sat between proj and block, post-proj negatives would be zeroed,
    # so the post-projection feature map would have NO negative entries. Catch the trap by
    # constructing z that yields some negative proj outputs and confirming the model uses them.
    with t.no_grad():
        h_raw = kids['proj'](z)
        assert (h_raw < 0).any(), 'projection output should have some negative entries for random z'
        # If a hidden ReLU lurked, replacing the projection with its ReLU version would NOT change the output:
        h_relu_view = F.relu(h_raw).view(B, base_c, base_hw, base_hw)
        out_if_relu = kids['block'](h_relu_view)
        assert not t.allclose(out, out_if_relu, atol=1e-6), (
            'output is identical whether or not we ReLU the projection — '
            'either you inserted a hidden ReLU between proj and block, or BN is masking it. '
            'The bottleneck must be a BARE Linear projection.'
        )
    _dd_passed.add('cx4')

_test_cx4()

<details><summary>Show solution — cx4</summary>

```python
def cx4_make_decoder_cls():
    class Decoder(nn.Module):
        def __init__(self, latent_dim, base_channels, base_hw):
            super().__init__()
            self.base_c = base_channels
            self.base_hw = base_hw
            # Atom A (bottleneck-latent-projection): bare Linear, no activation.
            self.proj = nn.Linear(latent_dim, base_channels * base_hw * base_hw)
            # Atom B (convtranspose-bn-activation-block): 2x upsample, channels halve.
            self.block = nn.Sequential(
                nn.ConvTranspose2d(base_channels, base_channels // 2,
                                   kernel_size=4, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(base_channels // 2),
                nn.ReLU(inplace=True),
            )

        def forward(self, z):
            B = z.size(0)
            h = self.proj(z).view(B, self.base_c, self.base_hw, self.base_hw)
            return self.block(h)

    return Decoder
```

Reshaping with `.view(B, C, H, W)` requires `B*C*H*W == proj.out_features`, which is enforced here by construction. Using `einops.layers.torch.Rearrange` instead (cx2) would let you skip the `forward` reshape entirely, at the cost of a tiny extra Module. The 'no activation at the bottleneck' rule is what lets the latent space have a principled probabilistic interpretation (especially in VAEs, where `z ~ N(mu, sigma**2)` must be allowed to take any real value).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["Generative: Bottleneck latent projection", "GAN: ConvT+BN+Activation block"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()